# Building an End-to-End Local AI Assistant

# 1. Project Overview

## 1.1 Project Objective

The objective of this project is to integrate the existing Local AI Assistant components into a usable, end-to-end application powered by a locally hosted Large Language Model (LLM).

The application will provide a unified interface for normal conversations, streaming generation, structured JSON responses, runtime status checks, and generation performance metrics.

Rather than implementing another isolated component, this project focuses on integrating existing modules into a coherent application with clearly defined responsibilities, reliable error handling, and testable behavior.

## 1.2 Learning Objectives

By completing this project, I aim to:

- Integrate the conversation service, prompt builder, runtime adapter, and structured output validator.
- Build an interactive command-line interface (CLI).
- Support both non-streaming and streaming generation.
- Validate structured responses using JSON Schema and Pydantic.
- Expose useful runtime information and generation metrics.
- Evaluate the existing prompt construction strategy using explicit test cases.
- Validate application behavior through automated tests and end-to-end scenarios.

## 1.3 Project Scope

The implementation will focus on the following capabilities:

1. Interactive command-line conversations.
2. Conversation context management.
3. Non-streaming text generation.
4. Streaming response display.
5. Structured JSON generation and validation.
6. Runtime health and model availability checks.
7. Generation performance metrics.
8. Error handling and graceful shutdown.
9. Automated testing and end-to-end validation.

The project will reuse the existing application components wherever possible and avoid unnecessary duplication.

## 1.4 Existing System Components

The application builds on the following components:

| Component | Responsibility |
|---|---|
| Runtime Configuration | Loads and validates application settings. |
| Conversation Manager | Maintains conversation messages and state. |
| Prompt Builder | Constructs the text prompt from conversation messages. |
| Conversation Service | Coordinates message processing and generation workflows. |
| Ollama Runtime | Communicates with the local Ollama API. |
| Generation Schemas | Validate generation requests, results, and streaming events. |
| Structured Output Module | Parses JSON and validates generated data. |
| Context Management | Tracks token usage and manages the conversation context window. |

The implementation will begin by verifying the existing interfaces and then integrate them into the end-to-end application.

## 1.5 Implementation Approach

The work will follow an incremental engineering workflow:

1. Inspect the existing implementation.
2. Define the application interface and integration points.
3. Implement one capability at a time.
4. Run focused tests after each meaningful change.
5. Inspect actual results and resolve failures.
6. Run end-to-end scenarios.
7. Document the final implementation, test results, and limitations.

All implementation artifacts and notebook content will be written in English.

## 1.6 Success Criteria

The project will be considered complete when:

- The CLI can start and shut down gracefully.
- Users can exchange messages with the local LLM.
- Conversation context is maintained correctly.
- Streaming responses are displayed incrementally.
- Structured responses are validated before being accepted.
- Runtime status and available metrics can be inspected.
- Expected runtime failures are handled appropriately.
- Automated tests pass, with any environment-dependent limitations documented.
- The notebook accurately reflects the implemented system and observed results.

# 2. Environment and Configuration

This section verifies the development environment, reviews the application configuration, checks the local Ollama runtime, and establishes a baseline for automated testing.

The purpose is to confirm that the existing system is ready for integration before introducing new application behavior.

## 2.1 Environment Verification

The project runs on Windows and uses the existing Python interpreter and installed project dependencies.

The verification process will confirm:

- The active Python version.
- The current working directory.
- The availability of the project's test framework.
- The successful import of the existing application modules.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

print(f"Python version: {sys.version.split()[0]}")
print(f"Python executable: {sys.executable}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Project root exists: {PROJECT_ROOT.is_dir()}")
print(f"Application directory exists: {(PROJECT_ROOT / 'app').is_dir()}")
print(f"Tests directory exists: {(PROJECT_ROOT / 'tests').is_dir()}")

Python version: 3.13.0
Python executable: c:\Users\mahmu\AppData\Local\Programs\Python\Python313\python.exe
Notebook directory: d:\Projects\Local AI Assistan\notebooks
Project root: d:\Projects\Local AI Assistan
Project root exists: True
Application directory exists: True
Tests directory exists: True


### Results and Interpretation

The environment verification completed successfully.

| Check | Observed Result | Status |
|---|---|---|
| Python Version | 3.13.0 | Passed |
| Python Executable | `C:\Users\mahmu\AppData\Local\Programs\Python\Python313\python.exe` | Passed |
| Notebook Directory | `D:\Projects\Local AI Assistan\notebooks` | Passed |
| Project Root | `D:\Projects\Local AI Assistan` | Passed |
| Project Root Exists | `True` | Passed |
| Application Directory Exists | `True` | Passed |
| Tests Directory Exists | `True` | Passed |

**Key Findings**

- The notebook is running with the expected Python 3.13.0 interpreter.
- The project root was correctly identified as the parent directory of the notebook directory.
- Both the application source directory and the test directory exist.
- The initial relative-path issue was resolved by deriving the project root from the notebook location.
- No changes to the Python interpreter or development environment were required.

**Conclusion**

The environment verification passed all checks. The notebook can now locate the existing application and test directories using the resolved project root.

**Next Step:** Inspect the effective runtime configuration before checking Ollama connectivity.

## 2.2 Runtime Configuration

The Local AI Assistant uses a centralized configuration system to manage runtime and generation settings.

The `RuntimeSettings` model validates configuration values, while `load_settings()` loads settings from the available configuration sources.

Configuration precedence is defined as follows:

1. Model defaults.
2. Values loaded from the `.env` file.
3. Operating-system environment variables.

The configuration includes the Ollama host, model name, request timeout, context window, generation length, sampling parameters, and model retention settings.

This section inspects the effective configuration without modifying any settings or initiating a model generation request.

#### Configuration Inspection

The following cell displays the active runtime settings. Sensitive values, if any, should not be printed.

In [2]:
import sys

PROJECT_ROOT = PROJECT_ROOT.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.config import load_settings

settings = load_settings()

print(f"Ollama host: {settings.ollama_host}")
print(f"Model name: {settings.model_name}")
print(f"Request timeout: {settings.request_timeout_seconds}s")
print(f"Context window: {settings.num_ctx}")
print(f"Maximum new tokens: {settings.max_new_tokens}")
print(f"Temperature: {settings.temperature}")
print(f"Top-k: {settings.top_k}")
print(f"Top-p: {settings.top_p}")
print(f"Seed: {settings.seed}")
print(f"Keep alive: {settings.keep_alive}")
print(f"Thinking configuration: {settings.think}")

Ollama host: http://127.0.0.1:11434
Model name: qwen3.8:27b
Request timeout: 120.0s
Context window: 4096
Maximum new tokens: 16
Temperature: 0.0
Top-k: None
Top-p: None
Seed: None
Keep alive: 5m
Thinking configuration: False


### Results and Interpretation

The runtime configuration was loaded successfully.

| Configuration | Observed Value |
|---|---|
| Ollama Host | `http://127.0.0.1:11434` |
| Model Name | `qwen3.8:27b` |
| Request Timeout | 120.0 seconds |
| Context Window | 4096 tokens |
| Maximum New Tokens | 16 |
| Temperature | 0.0 |
| Top-k | Not explicitly configured |
| Top-p | Not explicitly configured |
| Random Seed | Not explicitly configured |
| Keep Alive | `5m` |
| Thinking Configuration | `False` |

**Key Findings**

- The application is configured to communicate with a local Ollama instance.
- The configured model is `qwen3.8:27b`.
- The context window is set to 4096 tokens.
- The maximum generation length is currently set to 16 tokens.
- A temperature of `0.0` is configured for generation.
- Top-k, top-p, and seed are not explicitly configured in the effective settings.
- The model retention setting is `5m`.
- The thinking configuration is disabled.

**Engineering Observations**

The maximum generation length of 16 tokens may be insufficient for longer conversational responses or structured JSON outputs. Its practical impact will be evaluated during application testing rather than changing the configuration without evidence.

The absence of explicit top-k, top-p, and seed values means that these settings are not supplied through the effective configuration. Their final behavior depends on the runtime and model defaults.

**Conclusion**

The runtime configuration was loaded and inspected successfully. These values establish the configuration baseline for the upcoming Ollama connectivity and model availability checks.

No configuration changes were made during this inspection.

**Next Step:** Verify Ollama connectivity and confirm that the configured model is available.

## 2.3 Ollama Connectivity

This section verifies connectivity between the Local AI Assistant and the local Ollama runtime.

The application uses the existing `OllamaRuntime` adapter to communicate with the Ollama HTTP API.

The health check verifies that the Ollama version endpoint responds with a valid version identifier.

This check establishes runtime connectivity but does not, by itself, verify model availability or successful text generation.

### Verification Procedure

1. Initialize the existing runtime adapter using the validated application settings.
2. Execute the runtime health check.
3. Record the observed result.
4. Close the runtime client when it is no longer needed.

In [3]:
from app.runtime.ollama_runtime import OllamaRuntime

runtime = OllamaRuntime(settings)

try:
    is_healthy = runtime.health()
    print(f"Ollama health check: {'PASSED' if is_healthy else 'FAILED'}")
    print(f"Ollama host: {settings.ollama_host}")
finally:
    runtime.close()

Ollama health check: PASSED
Ollama host: http://127.0.0.1:11434


### Results and Interpretation

The Ollama connectivity check completed successfully.

| Check | Observed Result | Status |
|---|---|---|
| Health Check | `PASSED` | Passed |
| Ollama Host | `http://127.0.0.1:11434` | Passed |

#### Key Findings

- The application successfully connected to the configured local Ollama endpoint.
- The health check received a valid response from the Ollama version endpoint.
- The runtime adapter successfully validated the response and confirmed that Ollama was reachable.

#### Interpretation

The successful health check confirms that the local Ollama runtime is accessible through the configured host.

However, this result does not confirm that the configured model is available or that text generation works correctly. These capabilities require separate verification.

#### Conclusion

The Ollama connectivity check passed. The application can proceed to the model availability verification stage.

**Next Step:** Verify that `qwen3.8:27b` is available in the local Ollama model inventory.

## 2.4 Model Availability

This section verifies that the model configured for the Local AI Assistant is available in the local Ollama model inventory.

The application uses the existing `OllamaRuntime` adapter to retrieve the installed model identifiers through the Ollama API.

The verification will:

1. Retrieve the available model identifiers.
2. Compare the configured model name against the returned inventory.
3. Report whether the configured model is available.
4. Display the available model identifiers for inspection.

This check verifies model availability without initiating text generation or modifying the current runtime configuration.

In [4]:
# Retrieve the installed model identifiers from the local Ollama runtime.
runtime = OllamaRuntime(settings)

try:
    available_models = runtime.list_models()

    # Check whether the configured model is present in the inventory.
    model_available = settings.model_name in available_models

    # Display the configured model and its availability status.
    print(f"Configured model: {settings.model_name}")
    print(f"Model availability: {'PASSED' if model_available else 'FAILED'}")

    # Display all model identifiers returned by Ollama.
    print("\nAvailable models:")

    for model_name in available_models:
        print(f"- {model_name}")

finally:
    # Release the HTTP client resources after the inventory check.
    runtime.close()

Configured model: qwen3.8:27b
Model availability: PASSED

Available models:
- qwen3.8:27b


### Results and Interpretation

The model availability check completed successfully.

| Check | Observed Result | Status |
|---|---|---|
| Configured Model | `qwen3.8:27b` | Passed |
| Model Availability | `PASSED` | Passed |
| Available Models | `qwen3.8:27b` | Passed |

#### Key Findings

- The Ollama model inventory was retrieved successfully.
- The configured model, `qwen3.8:27b`, was found in the available model inventory.
- The configured model name matched an identifier returned by Ollama.

#### Interpretation

The successful availability check confirms that the configured model is registered in the local Ollama model inventory.

This result establishes that the application can identify the configured model through the Ollama API. However, model availability alone does not guarantee successful text generation or correct application behavior.

#### Conclusion

The model availability check passed. Both Ollama connectivity and configured model availability have now been verified.

**Next Step:** Review the baseline automated test results before beginning the end-to-end application integration.

## 2.5 Baseline Test Results

This section records the automated test results obtained before implementing the Day 5 end-to-end application.

The baseline provides a reference point for evaluating whether subsequent integration changes introduce regressions.

### Baseline Test Execution

The existing test suite was executed using the project's current Python environment:

```bash
python -m pytest .\tests -q
```
### Section Summary

The environment and runtime verification steps have been completed.

| Verification | Result |
|---|---|
| Python Environment | Python 3.13.0 |
| Project Root | Correctly identified |
| Application Directory | Available |
| Tests Directory | Available |
| Runtime Configuration | Loaded successfully |
| Ollama Connectivity | Passed |
| Configured Model Availability | Passed |
| Baseline Automated Tests | 212 passed |

### Key Observations

- The notebook can locate the application source code and test suite.
- The runtime configuration was loaded and inspected successfully.
- The local Ollama endpoint is reachable.
- The configured model, `qwen3.8:27b`, is available in the local model inventory.
- The existing automated test suite passed before Day 5 implementation began.
- The configured maximum generation length is 16 tokens and will be evaluated during integration.

### Conclusion

The initial environment and runtime checks passed, and the existing application provides a verified baseline for the next implementation phase.

No runtime configuration changes were made during these verification steps.

**Next Section:** End-to-End Application Design.

# 3. End-to-End Application Design

This section defines how the existing Local AI Assistant components will be integrated into a unified, interactive application.

The design focuses on reusing existing modules, maintaining clear separation of responsibilities, and providing reliable interaction with the local language model.

The implementation will build on the existing conversation service, prompt builder, runtime adapter, configuration system, and structured output validator.

## 3.1 Application Requirements

The integrated application should provide the following capabilities:

- Interactive command-line conversations.
- Conversation context management.
- Non-streaming text generation.
- Streaming response display.
- Structured JSON generation and validation.
- Runtime health and model availability checks.
- Generation performance metrics.
- Graceful shutdown and error handling.

The implementation must preserve the behavior of existing components and introduce new functionality without unnecessarily duplicating existing logic.

### Design Constraints

- Reuse the existing Python environment.
- Keep the existing Ollama runtime integration.
- Preserve the current configuration system.
- Validate new functionality through automated tests.
- Document actual implementation results rather than assuming success.

## 3.2 User Interaction Flow

The command-line interface (CLI) will provide a single entry point for interacting with the Local AI Assistant.

The application will accept user input, identify the requested operation, invoke the appropriate application component, and display the resulting output.

### Interaction Workflow

The expected interaction follows these steps:

1. Initialize the application and load the validated runtime configuration.
2. Create the runtime adapter and conversation service.
3. Display the available commands and prompt the user for input.
4. Identify whether the input represents a command or a conversation message.
5. Route conversation messages to the appropriate generation workflow.
6. Display the generated response or relevant operation status.
7. Continue accepting input until the user requests an exit.
8. Release application resources before shutting down.

### Interaction Modes

The CLI will support the following interaction modes:

| Mode | Purpose |
|---|---|
| Normal Chat | Generate a complete response and display it after generation finishes. |
| Streaming Chat | Display response fragments as they arrive from the runtime. |
| Structured Chat | Generate JSON output and validate it against the requested schema. |
| Runtime Status | Check Ollama connectivity and inspect available models. |
| Performance Metrics | Display available generation metrics when a generation result is available. |
| Exit | End the interactive session and release resources. |

### Command Design Principles

- Commands must be easy to understand and use.
- Invalid commands must produce clear feedback.
- Generation errors must not terminate the application unexpectedly.
- Conversation messages must be handled by the existing conversation service.
- Runtime operations must reuse the existing Ollama runtime adapter.
- Application resources must be released when the session ends.

The final command syntax will be determined after inspecting the existing conversation and runtime interfaces.

### Design Rationale

Separating commands from conversation messages makes the CLI easier to maintain and extend.

The CLI is responsible for user interaction and output presentation. The conversation service coordinates generation workflows, while the runtime adapter communicates with Ollama.

This separation keeps application orchestration independent of the underlying HTTP implementation and allows the existing components to be tested independently.

The interaction design remains provisional until it has been checked against the current application interfaces.

## 3.3 Component Integration

The Local AI Assistant is built from independent components with clearly defined responsibilities. This section describes how these components work together to support an end-to-end conversation workflow.

### 3.3.1 Conversation Manager

The `ConversationManager` class manages the conversation history in memory.

Its responsibilities include:

- Storing user and assistant messages in chronological order.
- Validating message roles and content through Pydantic models.
- Maintaining an optional system prompt separately from conversational messages.
- Returning conversation messages in their original order.
- Clearing conversational history while preserving the configured system prompt.
- Exposing the number of stored conversational messages.

The `ConversationMessage` model is immutable and validates message content before it is stored.

The conversation manager does not perform model inference or communicate directly with Ollama.

### 3.3.2 Prompt Builder

The prompt builder converts validated conversation messages into the text prompt expected by the current generation workflow.

It separates prompt construction from conversation storage and runtime communication.

The existing prompt formatting strategy will be retained unless testing provides evidence that a change is necessary.

### 3.3.3 Conversation Service

The conversation service coordinates the interaction between conversation management, prompt construction, runtime generation, and structured output validation.

It provides the existing workflows for normal message generation, streaming generation, and structured responses.

The CLI will use this service instead of duplicating conversation orchestration logic.

### 3.3.4 Ollama Runtime Adapter

The `OllamaRuntime` class communicates with the local Ollama HTTP API.

Its responsibilities include:

- Checking runtime health.
- Listing available models.
- Sending non-streaming generation requests.
- Processing streaming generation events.
- Normalizing runtime responses into validated application models.
- Exposing available generation metadata and client-observed latency.

The runtime adapter isolates HTTP communication from the higher-level application workflow.

### 3.3.5 Configuration and Validation

The configuration system loads and validates runtime settings before they are used by the application.

Pydantic models validate generation requests, runtime results, conversation messages, and structured responses.

These validation boundaries help detect invalid inputs and unexpected runtime responses.

### 3.3.6 Integration Flow

The expected interaction between the existing components is:

1. The application loads the runtime configuration.
2. The CLI accepts a user message.
3. The conversation service coordinates message processing.
4. The conversation manager provides the current conversation history.
5. The prompt builder constructs the model prompt.
6. The Ollama runtime adapter sends the generation request.
7. The runtime result is processed by the conversation service.
8. The validated assistant response is added to the conversation history.
9. The CLI displays the response to the user.

Streaming and structured generation follow their respective existing workflows while reusing the same underlying components.

#### Integration Principle

The implementation will reuse existing modules and preserve their responsibilities. New code will focus on application entry points, command handling, and integration rather than recreating functionality that already exists.

#### Design Interpretation

The existing architecture separates conversation state, prompt construction, generation orchestration, runtime communication, and data validation.

This separation provides a foundation for building the CLI without coupling user interaction directly to the Ollama HTTP API.

The `ConversationManager` maintains in-memory conversation history, while the conversation service coordinates generation workflows. The runtime adapter remains responsible for communication with Ollama.

The integration implementation must preserve these boundaries and validate the complete workflow through automated tests.

**Next Step:** Define the application's error handling strategy and resource lifecycle before implementing the CLI.

## 3.4 Error Handling Strategy

The Local AI Assistant uses explicit exception types to distinguish failures that occur during runtime communication and generation.

The existing exception hierarchy is defined in `app/runtime/exceptions.py`. It provides a shared base exception and specialized exceptions for connection failures, timeouts, HTTP errors, invalid runtime responses, missing models, streaming failures, and invalid generation requests.

Structured output processing defines a separate exception hierarchy in `app/structured_output.py`.

### 3.4.1 Runtime Error Classification

The existing runtime exceptions are classified as follows:

| Exception | Meaning | Proposed CLI Behavior |
|---|---|---|
| `RuntimeConnectionError` | The Ollama service cannot be reached. | Explain the connectivity problem and suggest checking the service. |
| `RuntimeTimeoutError` | A runtime request exceeded its configured timeout. | Inform the user that the request timed out. |
| `RuntimeHTTPError` | The runtime returned an unsuccessful HTTP status. | Display an appropriate error message and preserve diagnostic details when needed. |
| `RuntimeResponseError` | The runtime returned an invalid response. | Report that the response could not be processed. |
| `ModelNotFoundError` | The requested model is unavailable. | Identify the model availability problem. |
| `RuntimeStreamingError` | A streaming request could not be completed correctly. | Report the streaming failure and avoid treating an incomplete response as successful. |
| `InvalidGenerationRequestError` | A generation request violates the runtime interface requirements. | Handle the invalid operation without exposing unnecessary implementation details. |

These exception types already exist. The CLI behaviors described above are design requirements and will be implemented and tested during integration.

### 3.4.2 Structured Output Errors

The structured output module defines the following specialized exceptions:

| Exception | Meaning |
|---|---|
| `StructuredOutputParsingError` | Generated text could not be parsed as valid JSON. |
| `StructuredOutputSchemaError` | The supplied JSON Schema is invalid. |
| `StructuredOutputValidationError` | Parsed data does not satisfy the expected schema or model constraints. |

These errors are distinct from runtime communication errors because a generation request may succeed while its output fails parsing or validation.

### 3.4.3 Error Handling Principles

The application will follow these principles:

- **Explicit classification:** Handle known failure categories using their existing exception types.
- **Clear feedback:** Display understandable messages instead of exposing raw tracebacks for expected operational failures.
- **Session stability:** Allow the CLI to continue after recoverable operation failures when it is safe to do so.
- **Response integrity:** Do not treat incomplete or invalid generated output as a successful assistant response.
- **Resource cleanup:** Release runtime resources when the application shuts down, including when an exception occurs.
- **Diagnostic visibility:** Preserve useful error details for debugging without unnecessarily exposing internal implementation details to the user.

### 3.4.4 Exception Handling Boundaries

Error handling will be implemented at the appropriate application boundary.

The runtime adapter remains responsible for translating HTTP and streaming failures into runtime-specific exceptions.

The conversation service remains responsible for coordinating generation workflows and enforcing their validation requirements.

The CLI will handle expected application errors, display appropriate feedback, and manage the interactive session lifecycle.

This separation prevents user-interface logic from becoming tightly coupled to the underlying HTTP implementation.

### 3.4.5 Recovery and Retry Policy

The initial CLI will report recoverable errors and allow the user to decide whether to try again.

Automatic retries will not be introduced without a specific justification, because retrying can increase latency and may repeat expensive generation requests.

Invalid requests and validation failures will not be treated as successful operations. They must be corrected or handled explicitly before the workflow can continue.

#### Design Interpretation

The existing exception hierarchy provides the foundation for predictable error handling across the application.

The runtime adapter already distinguishes several operational failures. The structured output module separately identifies JSON parsing, schema definition, and data validation errors.

The main Day 5 integration task is to connect these existing error contracts to the CLI without duplicating exception definitions or moving runtime responsibilities into the user interface.

The proposed recovery behavior will be verified through implementation and testing rather than assumed to work in advance.

**Next Step:** Define the resource lifecycle and shutdown behavior before implementing the interactive CLI.

## 3.5 Design Decisions

This section documents the main engineering decisions guiding the end-to-end integration.

The decisions are based on the existing application architecture, the verified runtime configuration, and the requirements of the Day 5 implementation.

### 3.5.1 Reuse Existing Components

The implementation will reuse the existing application modules wherever possible.

These include:

- `ConversationManager` for in-memory conversation history.
- The prompt builder for constructing model prompts.
- `ConversationService` for coordinating generation workflows.
- `OllamaRuntime` for communication with the local Ollama API.
- Pydantic models for request, response, and message validation.
- The structured output module for JSON parsing and validation.
- The context management modules for token counting and context window management.

New components will be introduced only when needed to provide application-level functionality that does not already exist.

### 3.5.2 Preserve Environment and Configuration

The implementation will use the existing Python 3.13.0 interpreter and installed dependencies.

The current runtime configuration will remain unchanged unless testing demonstrates a concrete reason to modify it.

The configured model is `qwen3.8:27b`, and the current context window is 4096 tokens.

The maximum generation length is configured to 16 tokens. Its impact on conversational and structured responses will be evaluated during integration.

### 3.5.3 Maintain Separation of Responsibilities

Each application component will retain a clearly defined responsibility.

| Component | Responsibility |
|---|---|
| CLI | User input, command handling, and output presentation |
| Conversation Manager | In-memory conversation history |
| Prompt Builder | Prompt construction |
| Conversation Service | Generation workflow orchestration |
| Ollama Runtime | Runtime communication and response normalization |
| Structured Output Module | JSON parsing and response validation |
| Configuration Module | Loading and validating runtime settings |

This separation reduces unnecessary coupling and makes individual components easier to test.

### 3.5.4 Validate Incrementally

Development will follow an incremental implementation and verification process.

Each meaningful change will be followed by focused tests before the next feature is introduced.

After integration, the full automated test suite will be executed to identify regressions.

The existing baseline of 212 passing tests will be used as a reference for comparison.

### 3.5.5 Preserve Response Integrity

The application must distinguish successful generation from incomplete generation or invalid structured output.

A response must not be presented as a successful completed operation when the underlying workflow reports a failure.

Structured responses must pass the appropriate parsing and validation steps before they are accepted.

### 3.5.6 Measure Before Optimizing

Performance-related changes will be based on observed results rather than assumptions.

The application will reuse available runtime metadata, including token counts, runtime durations, and client-observed latency.

The implementation will distinguish runtime-reported measurements from client-side observations.

### 3.5.7 Keep Design Changes Evidence-Based

The current prompt construction strategy and runtime interfaces will be retained unless testing reveals a specific limitation.

Changes must address an identified problem, preserve existing behavior where appropriate, and be validated through automated tests.

This approach avoids unnecessary refactoring during the end-to-end integration.

#### Design Decision Summary

The implementation will focus on integrating existing components into a usable application rather than rebuilding the underlying generation infrastructure.

The implementation will prioritize:

1. Reuse of existing functionality.
2. Clear separation of responsibilities.
3. Reliable handling of successful and failed operations.
4. Incremental testing and regression prevention.
5. Evidence-based performance evaluation.
6. Accurate documentation of implementation results.

These decisions establish the engineering constraints for the upcoming implementation phase.

**Next Step:** Begin the implementation by establishing the CLI foundation and defining its entry point.

# 4. Interactive CLI Foundation

## 4.1 CLI Architecture

The command-line interface (CLI) provides the primary interaction layer between the user and the local AI assistant.

Instead of requiring users to call application services directly, the CLI translates user input into application operations and presents the resulting responses in a readable format.

The implementation is organized around three responsibilities:

- **Dependency initialization:** Create the runtime and conversation service using the existing application configuration.
- **Command routing:** Recognize supported commands and direct user requests to the appropriate service method.
- **User interaction:** Display responses, report operational errors, and maintain the interactive session until the user exits.

## 4.2 Supported Interaction Modes

The CLI integrates several capabilities exposed by the existing application components:

| Interaction | Purpose |
|---|---|
| Normal chat | Generate a complete response before displaying it. |
| Streaming chat | Display generated text incrementally. |
| Structured output | Request and validate a response against a defined schema. |
| Runtime status | Check connectivity and configured model availability. |
| Conversation statistics | Display message counts and basic conversation information. |
| Exit | Terminate the interactive session cleanly. |

## 4.3 Separation of Responsibilities

The CLI does not implement model generation or structured-output validation itself.

Instead, it delegates these responsibilities to the existing application layers:

- `ConversationManager` manages conversation messages.
- `PromptBuilder` constructs the model prompt.
- `ConversationService` coordinates application-level operations.
- `OllamaRuntime` communicates with the local Ollama API.
- `RuntimeSettings` supplies runtime configuration.

This separation keeps the interaction layer relatively small and allows the underlying components to be tested independently.

## 4.4 Validation Strategy

The CLI foundation is validated at multiple levels:

1. Python syntax compilation checks whether the module can be compiled.
2. Unit tests verify command routing, user interaction, and error handling.
3. Integration tests verify actual communication with the configured local model.

Passing one validation level does not automatically establish that all other levels have passed.

In [5]:
# Import the CLI entry point and dependency factory.
from app.cli import create_conversation_service, run_cli

# Confirm that both required functions are available.
assert callable(create_conversation_service)
assert callable(run_cli)

print("CLI module import: PASSED")
print("Dependency factory: AVAILABLE")
print("Interactive entry point: AVAILABLE")

CLI module import: PASSED
Dependency factory: AVAILABLE
Interactive entry point: AVAILABLE


### Results and Interpretation

The CLI module was imported successfully, and both required entry points were available.

- **CLI module import:** Passed.
- **Dependency factory:** Available.
- **Interactive entry point:** Available.

These results confirm that the CLI module can be imported and that its main entry points are accessible. This validates the module's basic importability, but does not independently verify command execution or interaction with the local model.

The next validation stage focuses on the normal chat workflow.

# 5. Normal Chat Integration

## 5.1 Request Lifecycle

Normal chat provides the standard request-response interaction with the local language model.

The workflow follows these steps:

1. The user enters a message through the CLI.
2. The CLI forwards the message to `ConversationService.send_message()`.
3. The conversation service validates the input and updates the conversation state.
4. `PromptBuilder` constructs the prompt from the conversation messages.
5. `OllamaRuntime` sends the generation request to the configured local model.
6. The service receives the generation result and stores the assistant response.
7. The CLI displays the completed response.

## 5.2 Response Handling

Normal chat waits for the generation operation to complete before displaying the final response.

This behavior differs from streaming generation, where text is delivered incrementally as generation progresses.

The normal chat path is useful when the application needs a complete response object before presenting the result.

## 5.3 Error Handling

The CLI handles runtime-related failures through the application's exception hierarchy.

Errors can arise from connection failures, request timeouts, HTTP errors, invalid runtime responses, or model availability problems.

The interface should report these failures without presenting an unsuccessful generation as a successful assistant response.

## 5.4 Validation Approach

The normal chat workflow should be evaluated at two levels:

- **Automated validation:** Unit tests verify service behavior and the CLI's handling of normal chat requests.
- **Runtime validation:** An actual request to the configured local model confirms that the components work together in the current environment.

The distinction matters because successful unit tests do not, by themselves, establish that a real model request succeeds.

In [ ]:
# Import the existing application components.
from pathlib import Path

from app.config import load_settings
from app.context.token_counter import TokenCounter
from app.context.window_manager import ContextWindowManager
from app.conversation import ConversationManager
from app.prompt_builder import PromptBuilder
from app.runtime.ollama_runtime import OllamaRuntime
from app.services.conversation_service import ConversationService

# Load the project's current runtime configuration.
settings = load_settings()

# Resolve the tokenizer file relative to the project root.
project_root = Path.cwd().parent
tokenizer_path = (
    project_root / "assets" / "tokenizers" / "qwen3.8-27b" / "tokenizer.json"
)

# Initialize the runtime and conversation components.
runtime = OllamaRuntime(settings)

try:
    conversation = ConversationManager()
    prompt_builder = PromptBuilder()
    token_counter = TokenCounter(tokenizer_path)

    # Initialize context-window management.
    context_window_manager = ContextWindowManager(
        prompt_builder=prompt_builder,
        token_counter=token_counter,
    )

    # Create the conversation service using its actual constructor.
    service = ConversationService(
        runtime=runtime,
        conversation=conversation,
        prompt_builder=prompt_builder,
        context_window_manager=context_window_manager,
        settings=settings,
    )

    # Send a real non-streaming request to the configured local model.
    response = service.send_message("Explain tokenization in exactly two sentences.")

    # Display the generated response and its actual runtime fields.
    print("Normal Chat: COMPLETED")
    print(f"Configured model: {settings.model_name}")
    print(f"Response:\n{response.text}")
    print(f"Response length: {len(response.text)} characters")

    print("\nGeneration metadata:")
    for field, value in response.model_dump().items():
        print(f"{field}: {value}")

finally:
    # Release the HTTP client even if the request fails.
    runtime.close()

Normal Chat: COMPLETED
Configured model: qwen3.8:27b
Response:
Tokenization is the process of breaking down text into smaller units, such as words
Response length: 83 characters

Generation metadata:
text: Tokenization is the process of breaking down text into smaller units, such as words
model: qwen3.8:27b
done: True
done_reason: length
prompt_eval_count: 27
eval_count: 16
total_duration_ns: 2864926200
load_duration_ns: 14024900
prompt_eval_duration_ns: 850493000
eval_duration_ns: 1990972000
client_latency_ms: 2886.8624000024283


### Results and Interpretation

The normal chat request completed successfully against the local `qwen3.8:27b` model. However, the generated response did not satisfy the requested two-sentence format.

**Observed results:**

| Metric | Observed Value |
|---|---|
| Model | `qwen3.8:27b` |
| Generation completed | `True` |
| Completion reason | `length` |
| Prompt tokens evaluated | 27 |
| Output tokens generated | 16 |
| Response length | 83 characters |
| Total runtime duration | Approximately 10.19 seconds |
| Client-observed latency | Approximately 10.20 seconds |
| Output evaluation duration | Approximately 2.11 seconds |

**Analysis:**

The `done_reason` field reported `length`, and the runtime reported 16 generated tokens. These observations are consistent with the configured generation limit of 16 tokens.

Although the runtime marked generation as complete, the response ended before the requested explanation was complete. This demonstrates that successful request execution and successful completion according to the runtime do not necessarily mean that the answer satisfies the user's requirements.

The client-observed latency was approximately 10.20 seconds, while the reported output evaluation duration was approximately 2.11 seconds. These measurements describe different timing intervals and should not be treated as interchangeable.

**Conclusion:**

The normal chat path works, but the current generation limit is too restrictive for this request. The next experiment will increase the output token budget and compare the resulting response and runtime metadata.

## 5.5 Controlled Generation Experiment

A second normal chat experiment was conducted using the same configured local model and the same prompt, but with a larger output token budget.

The original project configuration was not modified. The higher token limit was applied only to the experimental request.

In [ ]:
# Import the components required for a controlled generation experiment.
from pathlib import Path

from app.config import load_settings
from app.context.token_counter import TokenCounter
from app.context.window_manager import ContextWindowManager
from app.conversation import ConversationManager
from app.prompt_builder import PromptBuilder
from app.runtime.ollama_runtime import OllamaRuntime
from app.services.conversation_service import ConversationService

# Load the original project settings without modifying them.
settings = load_settings()

# Locate the tokenizer file used by the project.
project_root = Path.cwd().parent
tokenizer_path = (
    project_root / "assets" / "tokenizers" / "qwen3.8-27b" / "tokenizer.json"
)

# Initialize the runtime and supporting components.
runtime = OllamaRuntime(settings)

try:
    conversation = ConversationManager()
    prompt_builder = PromptBuilder()
    token_counter = TokenCounter(tokenizer_path)

    context_window_manager = ContextWindowManager(
        prompt_builder=prompt_builder,
        token_counter=token_counter,
    )

    service = ConversationService(
        runtime=runtime,
        conversation=conversation,
        prompt_builder=prompt_builder,
        context_window_manager=context_window_manager,
        settings=settings,
    )

    # Increase the output budget for this request only.
    response = service.send_message(
        "Explain tokenization in exactly two sentences.",
        max_new_tokens=128,
    )

    # Display the actual response and runtime-reported measurements.
    print("Normal Chat Experiment: COMPLETED")
    print(f"Configured model: {settings.model_name}")
    print(f"Requested output limit: 128 tokens")
    print(f"Response:\n{response.text}")
    print(f"Response length: {len(response.text)} characters")

    print("\nGeneration metadata:")
    for field, value in response.model_dump().items():
        print(f"{field}: {value}")

finally:
    # Ensure the runtime is closed after the experiment.
    runtime.close()

Normal Chat Experiment: COMPLETED
Configured model: qwen3.8:27b
Requested output limit: 128 tokens
Response:
Tokenization is the process of breaking down text into smaller units, such as words, subwords, or characters, that a machine learning model can process. These discrete tokens serve as the fundamental building blocks for natural language processing tasks like translation and text generation.
Response length: 291 characters

Generation metadata:
text: Tokenization is the process of breaking down text into smaller units, such as words, subwords, or characters, that a machine learning model can process. These discrete tokens serve as the fundamental building blocks for natural language processing tasks like translation and text generation.
model: qwen3.8:27b
done: True
done_reason: stop
prompt_eval_count: 27
eval_count: 52
total_duration_ns: 19561401800
load_duration_ns: 5767700
prompt_eval_duration_ns: 594438000
eval_duration_ns: 10840649000
client_latency_ms: 19580.095500001335


### Results and Interpretation

**Observed results:**

| Metric | Initial Experiment | Controlled Experiment |
|---|---:|---:|
| Model | `qwen3.8:27b` | `qwen3.8:27b` |
| Requested output limit | 16 tokens (configured default) | 128 tokens |
| Output tokens generated | 16 | 52 |
| Response length | 83 characters | 291 characters |
| Completion reason | `length` | `stop` |
| Total runtime duration | 10.19 seconds | 19.56 seconds |
| Client-observed latency | 10.20 seconds | 19.58 seconds |
| Output evaluation duration | 2.11 seconds | 10.84 seconds |

**Analysis:**

In the initial experiment, generation stopped at the configured 16-token limit, and the response did not satisfy the requested two-sentence format.

In the controlled experiment, the model generated 52 tokens and returned a complete two-sentence explanation. The runtime reported `stop` as the completion reason.

The results support the conclusion that the original output limit was insufficient for this request. The larger limit allowed the model to complete its response, although the request took longer.

The configured default remains unchanged. This experiment demonstrates how request-level generation options can be used to investigate response completeness without modifying the application's base configuration.

**Conclusion:**

Generation limits should be selected according to the expected response length. A successful runtime completion does not, by itself, guarantee that the response satisfies every content requirement; the returned text must also be inspected.

# 6. Streaming Generation

## 6.1 Objective

The objective of this section is to integrate and evaluate streaming generation through the existing local AI assistant.

Unlike normal chat, which returns the completed response after generation finishes, streaming delivers generated text incrementally as it becomes available.

## 6.2 What We Will Implement and Evaluate

This section focuses on the following objectives:

- **Streaming integration:** Use the existing `ConversationService.stream_message()` method to send a streaming request to the local Ollama model.
- **Incremental output:** Collect the text fragments produced during generation and reconstruct the final response.
- **Time to First Text:** Measure the elapsed time between starting the request and receiving the first non-empty text fragment.
- **Total elapsed time:** Measure the total client-observed duration of the streaming request.
- **Stream event inspection:** Count the events received during the streaming interaction.
- **Generation metadata:** Inspect the final generation result when it is available, including the completion reason, generated token count, and runtime timing measurements.

## 6.3 Experimental Approach

The experiment will use the configured `qwen3.8:27b` model through the existing application components.

The request will use a larger output token budget to reduce the likelihood of truncation during the experiment. This override applies only to the experimental request and does not modify the project's default configuration.

The experiment will record the actual response, stream events, and available timing measurements. No performance results will be assumed before the request has been executed.

## 6.4 Evaluation Criteria

The streaming workflow will be evaluated using the following criteria:

1. The streaming request completes without an application-level error.
2. Generated text is received incrementally and reconstructed into the final response.
3. Time to First Text and total client-observed elapsed time are measured.
4. The final generation metadata is inspected when available.
5. The response and completion metadata are examined to determine whether generation finished normally or was truncated.

These observations will provide the basis for comparing streaming generation with the normal chat workflow in a later section.

In [ ]:
# Import the components required for the streaming experiment.
from pathlib import Path
from time import perf_counter

from app.config import load_settings
from app.context.token_counter import TokenCounter
from app.context.window_manager import ContextWindowManager
from app.conversation import ConversationManager
from app.prompt_builder import PromptBuilder
from app.runtime.ollama_runtime import OllamaRuntime
from app.services.conversation_service import ConversationService

# Load the existing project configuration.
settings = load_settings()

# Resolve the tokenizer file relative to the project root.
project_root = Path.cwd().parent
tokenizer_path = (
    project_root / "assets" / "tokenizers" / "qwen3.8-27b" / "tokenizer.json"
)

# Initialize the runtime.
runtime = OllamaRuntime(settings)

try:
    # Initialize the conversation and prompt components.
    conversation = ConversationManager()
    prompt_builder = PromptBuilder()
    token_counter = TokenCounter(tokenizer_path)

    context_window_manager = ContextWindowManager(
        prompt_builder=prompt_builder,
        token_counter=token_counter,
    )

    # Create the conversation service.
    service = ConversationService(
        runtime=runtime,
        conversation=conversation,
        prompt_builder=prompt_builder,
        context_window_manager=context_window_manager,
        settings=settings,
    )

    # Record the time immediately before starting the request.
    request_start = perf_counter()
    first_text_time = None
    text_parts = []
    event_count = 0
    final_result = None

    # Request a two-sentence explanation with a larger output budget.
    for event in service.stream_message(
        "Explain tokenization in exactly two sentences.",
        max_new_tokens=128,
    ):
        event_count += 1

        # Collect each non-empty text fragment.
        if event.text_delta:
            if first_text_time is None:
                first_text_time = perf_counter()

            text_parts.append(event.text_delta)

            # Display the fragment immediately to demonstrate streaming.
            print(event.text_delta, end="", flush=True)

        # Capture metadata from the final streaming event.
        if event.is_final:
            final_result = event.final_metadata

    # Record the total client-observed elapsed time.
    request_end = perf_counter()
    collected_text = "".join(text_parts)

    print("\n\nStreaming Experiment: COMPLETED")
    print(f"Configured model: {settings.model_name}")
    print(f"Response length: {len(collected_text)} characters")
    print(f"Stream events received: {event_count}")

    if first_text_time is not None:
        print(f"Time to first text: {(first_text_time - request_start) * 1000:.2f} ms")
    else:
        print("Time to first text: No text event received")

    print(
        "Total client-observed elapsed time: "
        f"{(request_end - request_start) * 1000:.2f} ms"
    )

    # Display the final generation metadata when available.
    if final_result is not None:
        print("\nFinal generation metadata:")
        for field, value in final_result.model_dump().items():
            print(f"{field}: {value}")

finally:
    # Close the runtime after the streaming experiment.
    runtime.close()

Tokenization is the process of breaking down text into smaller units, such as words, subwords, or characters, that a machine learning model can process. These discrete tokens serve as the fundamental building blocks for natural language processing tasks like translation and text generation.

Streaming Experiment: COMPLETED
Configured model: qwen3.8:27b
Response length: 291 characters
Stream events received: 52
Time to first text: 9403.10 ms
Total client-observed elapsed time: 20245.98 ms

Final generation metadata:
text: Tokenization is the process of breaking down text into smaller units, such as words, subwords, or characters, that a machine learning model can process. These discrete tokens serve as the fundamental building blocks for natural language processing tasks like translation and text generation.
model: qwen3.8:27b
done: True
done_reason: stop
prompt_eval_count: 27
eval_count: 52
total_duration_ns: 20227999100
load_duration_ns: 3143300
prompt_eval_duration_ns: 637254000
eval

### Results and Interpretation

The streaming experiment completed successfully using the local `qwen3.8:27b` model.

**Observed results:**

| Metric | Observed Value |
|---|---:|
| Model | `qwen3.8:27b` |
| Generation completed | `True` |
| Completion reason | `stop` |
| Prompt tokens evaluated | 27 |
| Output tokens generated | 52 |
| Response length | 291 characters |
| Stream events received | 52 |
| Time to first text | 9,403.10 ms |
| Total client-observed elapsed time | 20,245.98 ms |
| Runtime-reported client latency | 20,245.07 ms |
| Output evaluation duration | 10.84 seconds |

**Response integrity:**

The reconstructed response contained 291 characters and matched the final text reported by the runtime. The service completed its stream validation and returned the final event with valid completion metadata.

The runtime reported `stop` as the completion reason, indicating that generation ended through a stop condition rather than the output-length limit reported in the earlier normal chat experiment.

**Latency observations:**

The first text fragment arrived approximately 9.40 seconds after the request started. The total client-observed elapsed time was approximately 20.25 seconds.

These measurements distinguish the initial wait for visible output from the total time required to complete generation. Streaming delivered text incrementally, but the first visible output still required a substantial initial wait.

The time-to-first-text measurement is client-observed and includes the effects of request setup, prompt processing, runtime behavior, and event delivery. It should not be interpreted as pure model inference time.

**Conclusion:**

The streaming workflow successfully delivered incremental text and completed response validation. The experiment also established measurable baseline values for first-text latency and total elapsed time.

### 6.5 Normal Chat vs. Streaming

The two experiments used the same model and prompt, with an output budget of 128 tokens.

The normal chat experiment returned the completed response after approximately 19.58 seconds of client-observed latency.

The streaming experiment delivered text incrementally, with the first text fragment arriving after approximately 9.40 seconds. The complete stream finished after approximately 20.25 seconds.

| Measurement | Normal Chat | Streaming |
|---|---:|---:|
| Model | `qwen3.8:27b` | `qwen3.8:27b` |
| Requested output limit | 128 tokens | 128 tokens |
| Output tokens generated | 52 | 52 |
| Response length | 291 characters | 291 characters |
| Completion reason | `stop` | `stop` |
| Client-observed latency | 19.58 seconds | 20.25 seconds |
| Time to first text | Not separately measured | 9.40 seconds |

**Interpretation:**

Both experiments produced the same response content and reported the same generated token count and completion reason.

The streaming experiment exposed intermediate text before the full response was available. However, its measured total elapsed time was approximately 0.67 seconds longer than the normal chat measurement.

This difference should not be interpreted as proof that streaming is inherently slower. These were individual requests rather than repeated, controlled benchmark trials, and execution time can vary between runs.

**Key takeaway:**

Streaming primarily changes how generated text is delivered to the user. Its main advantage is incremental visibility, while total generation time remains a separate performance metric.

# 7. Structured Output Integration

## 7.1 Objective

The objective of this section is to integrate and evaluate structured JSON generation using the existing local AI assistant.

Unlike normal chat, which returns free-form text, structured output requests a response that follows a predefined data contract.

The application uses Ollama's structured generation capabilities together with JSON Schema and Pydantic validation to constrain and validate the generated response.

## 7.2 Structured Response Contract

The project defines the expected assistant response through the `AssistantStructuredResponse` Pydantic model.

The model contains two required fields:

| Field | Type | Constraint |
|---|---|---|
| `answer` | String | Must contain at least one character |
| `confidence` | Float | Must be between `0.0` and `1.0`, inclusive |

The model also forbids additional fields, enforces strict type validation, and disallows non-finite floating-point values.

## 7.3 Structured Generation Workflow

The structured generation workflow follows these steps:

1. The application receives a user message.
2. The CLI enables structured JSON mode.
3. `AssistantStructuredResponse.model_json_schema()` produces the response schema.
4. `ConversationService.send_structured_message()` sends the request through the existing runtime.
5. The generated text is parsed as JSON.
6. The parsed data is validated against the supplied JSON Schema.
7. The service validates the response against the Pydantic model.
8. The validated assistant response is stored in the conversation and returned to the CLI.

This workflow separates generation from validation. A response is not accepted merely because the model produced text that resembles JSON.

## 7.4 Validation Objectives

The experiment will evaluate whether the local model can:

- Generate syntactically valid JSON.
- Include the required `answer` and `confidence` fields.
- Return values that satisfy the defined schema and Pydantic constraints.
- Complete the request without a parsing or validation error.
- Preserve the validated response in the conversation history.

The real-model experiment will be followed by a separate negative validation test to examine how the application handles invalid structured data.

In [ ]:
# Import the existing application components.
from pathlib import Path

from app.config import load_settings
from app.context.token_counter import TokenCounter
from app.context.window_manager import ContextWindowManager
from app.conversation import ConversationManager
from app.prompt_builder import PromptBuilder
from app.runtime.ollama_runtime import OllamaRuntime
from app.services.conversation_service import ConversationService
from app.structured_output import AssistantStructuredResponse

# Load the existing project configuration.
settings = load_settings()

# Resolve the tokenizer file relative to the project root.
project_root = Path.cwd().parent
tokenizer_path = (
    project_root / "assets" / "tokenizers" / "qwen3.8-27b" / "tokenizer.json"
)

# Initialize the runtime and supporting components.
runtime = OllamaRuntime(settings)

try:
    conversation = ConversationManager()
    prompt_builder = PromptBuilder()
    token_counter = TokenCounter(tokenizer_path)

    # Initialize context-window management.
    context_window_manager = ContextWindowManager(
        prompt_builder=prompt_builder,
        token_counter=token_counter,
    )

    # Create the conversation service.
    service = ConversationService(
        runtime=runtime,
        conversation=conversation,
        prompt_builder=prompt_builder,
        context_window_manager=context_window_manager,
        settings=settings,
    )

    # Generate the JSON Schema from the project's Pydantic model.
    response_schema = AssistantStructuredResponse.model_json_schema()

    # Send a real structured generation request to the local model.
    parsed_response = service.send_structured_message(
        "Explain tokenization in two sentences.",
        schema=response_schema,
        max_new_tokens=256,
    )

    # Validate the returned data against the Pydantic response model.
    validated_response = AssistantStructuredResponse.model_validate(parsed_response)

    # Display the validated response and its field values.
    print("Structured Output Experiment: COMPLETED")
    print(f"Configured model: {settings.model_name}")
    print("\nValidated JSON:")
    print(validated_response.model_dump_json(indent=2))
    print("\nValidation checks:")
    print(f"Answer is non-empty: {bool(validated_response.answer.strip())}")
    print(f"Confidence is within range: {0.0 <= validated_response.confidence <= 1.0}")
    # Report the number of stored conversation messages.
    print(f"Conversation message count: {conversation.message_count}")

finally:
    # Release runtime resources after the experiment.
    runtime.close()

Structured Output Experiment: COMPLETED
Configured model: qwen3.8:27b

Validated JSON:
{
  "answer": "Tokenization is the process of breaking down text into smaller units called tokens, such as words, subwords, or characters, which serve as the basic building blocks for machine learning models. This step allows algorithms to process natural language by converting human-readable text into numerical representations that can be understood and analyzed by computers.",
  "confidence": 0.95
}

Validation checks:
Answer is non-empty: True
Confidence is within range: True
Conversation message count: 2


### Results and Interpretation

The structured output request returned a JSON response from the local `qwen3.8:27b` model.

**Observed results:**

- The generated response contained the required `answer` and `confidence` fields.
- The confidence value was `0.95`.
- The answer was non-empty.
- The confidence value satisfied the configured range of `[0.0, 1.0]`.
- The response was parsed and validated through the application's structured output workflow.

The initial notebook cell encountered an `AttributeError` when attempting to access a nonexistent `messages` attribute. Inspection of the actual `ConversationManager` implementation identified `message_count` and `get_messages()` as the supported interfaces.

The error occurred during the final reporting step, after the structured response and validation checks had been printed. It did not indicate a structured generation or validation failure.

**Conclusion:**

The observed response satisfied the displayed validation checks. The next experiment evaluates whether the validation layer rejects invalid structured data.

## 7.5 Negative Validation Testing

### Objective

The objective of this experiment is to verify that the structured output validation layer rejects data that violates the application's response contract.

A response may be valid JSON while still containing missing fields, invalid values, or unexpected properties. Therefore, successful JSON parsing alone is insufficient to establish that a response is valid.

#### Test Cases

Three invalid inputs will be evaluated against the existing `AssistantStructuredResponse` Pydantic model.

| Test Case | Invalid Condition | Expected Behavior |
|---|---|---|
| Missing confidence | The required `confidence` field is absent. | Reject the response. |
| Confidence out of range | The `confidence` value is greater than `1.0`. | Reject the response. |
| Unexpected field | The JSON contains an additional `extra` field. | Reject the response. |

#### Validation Approach

The experiment uses the existing `parse_and_validate_model()` function with controlled JSON inputs.

Each input is processed locally without calling the language model. This isolates Pydantic validation behavior from model generation and avoids introducing runtime latency into the experiment.

The test checks whether each invalid input raises `StructuredOutputValidationError`. If an invalid input is accepted, the test will report that outcome explicitly.

#### Evaluation Criteria

The validation layer will be considered successful for this experiment if all three invalid inputs are rejected with `StructuredOutputValidationError`.

The actual execution results will be recorded in the following section after the test has run.

In [ ]:
# Import the project's structured output validation components.
from app.structured_output import (
    AssistantStructuredResponse,
    StructuredOutputValidationError,
    parse_and_validate_model,
)

# Define JSON inputs that violate the response contract.
invalid_cases = {
    "Missing confidence": ('{"answer": "Tokenization splits text into tokens."}'),
    "Confidence out of range": ('{"answer": "Valid answer.", "confidence": 1.5}'),
    "Unexpected field": (
        '{"answer": "Valid answer.", "confidence": 0.95, "extra": "value"}'
    ),
}

# Verify that each invalid response is rejected.
for case_name, json_text in invalid_cases.items():
    try:
        parse_and_validate_model(
            json_text,
            AssistantStructuredResponse,
        )
    except StructuredOutputValidationError as error:
        print(f"{case_name}: REJECTED")
        print(f"Validation error: {error}\n")
    else:
        print(f"{case_name}: UNEXPECTEDLY ACCEPTED\n")

Missing confidence: REJECTED
Validation error: Pydantic validation failed: 1 validation error for AssistantStructuredResponse
confidence
  Field required [type=missing, input_value={'answer': 'Tokenization ...lits text into tokens.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

Confidence out of range: REJECTED
Validation error: Pydantic validation failed: 1 validation error for AssistantStructuredResponse
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal

Unexpected field: REJECTED
Validation error: Pydantic validation failed: 1 validation error for AssistantStructuredResponse
extra
  Extra inputs are not permitted [type=extra_forbidden, input_value='value', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden



### Results and Interpretation

All three negative validation tests rejected their invalid inputs as expected.

**Observed results:**

| Test Case | Actual Result | Validation Reason |
|---|---|---|
| Missing confidence | REJECTED | Required field `confidence` was missing. |
| Confidence out of range | REJECTED | The value `1.5` exceeded the maximum allowed value of `1.0`. |
| Unexpected field | REJECTED | The additional `extra` field was forbidden. |

**Analysis:**

The missing-field test confirmed that the response contract requires the `confidence` field.

The range-validation test confirmed that values outside the permitted confidence interval are rejected.

The unexpected-field test confirmed that the model's `extra="forbid"` configuration prevents undeclared properties from being accepted.

All three cases raised `StructuredOutputValidationError`, demonstrating that the application's validation layer consistently rejects these categories of invalid data.

These tests validate the Pydantic model's behavior using controlled JSON inputs. They do not independently evaluate the language model's ability to generate valid structured responses.

**Conclusion:**

The structured output validation layer correctly rejected all three tested violations of the response contract. Combined with the successful real-model experiment, these results demonstrate both successful structured response handling and rejection of the tested invalid inputs.

## 7.6 Final Evaluation

The structured output workflow was evaluated through real-model generation and controlled negative validation tests.

The real-model experiment produced a JSON response containing the required `answer` and `confidence` fields. The response passed through the application's JSON Schema and Pydantic validation workflow and was stored in the conversation.

The negative tests independently verified that the Pydantic model rejected:

- Missing required fields.
- Confidence values outside the permitted range.
- Unexpected additional fields.

Together, these experiments demonstrate that structured output requires both successful generation and explicit validation.

A response that looks like JSON is not necessarily a valid application response. The generated data must also satisfy the expected schema and type constraints before it is accepted.

**Key takeaway:** Reliable structured generation depends on a clear response contract, runtime schema guidance, and explicit validation before the result is used by the application.